## Pursuit evasion game

# Check degli stati

In [1]:
import gc
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import importlib
import sys

import hj_reachability as hj


# ============================================================
# CONFIGURATION
# ============================================================

# Notebook assumed to be inside examples/
ROOT = Path.cwd().resolve().parent
BRT_DIRECTORY = ROOT / "results" / "brt"

# Exact filenames without .npz.
METRICS = [
    "euclidean",
    "ttc",
    "dce",
]

# ============================================================
# INITIAL STATE
# ============================================================

state = np.array([
    6.0,  # x_rel [m]
    0,  # y_rel [m]
    0,  # theta_rel [rad]
    5.0,  # v_H [m/s]
    0.0,  # delta_E [rad]
    8.0,  # v_E [m/s]
])

results = []

for metric in METRICS:
    brt_path = BRT_DIRECTORY / f"{metric}.npz"

    with np.load(
        brt_path,
        allow_pickle=True
    ) as data:
        grid_lo = np.asarray(
            data["grid_lo"], dtype=float,
        )
        grid_hi = np.asarray(
            data["grid_hi"], dtype=float,
        )
        grid_shape = tuple(
            int(value)
            for value in data["grid_shape"]
        )
        periodic_dims = tuple(
            int(value)
            for value in data["periodic_dims"]
        )
        grid = (
            hj.Grid.from_lattice_parameters_and_boundary_conditions(
                domain = hj.sets.Box(
                    lo=jnp.asarray(grid_lo),
                    hi=jnp.asarray(grid_hi),
                ),
                shape=grid_shape,
                periodic_dims=periodic_dims,
            )
        )
        V0 = jnp.asarray(data["V0"])
        BRT = jnp.asarray(data["BRT"])

        state_jax = jnp.asarray(
            state,
            dtype=float,
        )

        V0_interp = float(
            grid.interpolate(V0, state_jax)
        )
        BRT_interp = float(
            grid.interpolate(BRT, state_jax)
        )

        results.append({
            "metric": metric,
            "V0" : V0_interp,
            "BRT" : BRT_interp,
        })
        del grid, V0, BRT, state_jax

        gc.collect()
        jax.clear_caches()

for result in results:
    print(f"Metric: {result['metric']}")
    print(f"V0: {result['V0']}")
    print(f"BRT: {result['BRT']}\n")





Metric: euclidean
V0: 1.5044482946395874
BRT: -1.5475772619247437

Metric: ttc
V0: 0.5426419377326965
BRT: -0.8910059332847595

Metric: dce
V0: -1.342403531074524
BRT: -1.693295955657959



# BRT positivo

In [2]:
import gc
import importlib
import sys
from pathlib import Path

import jax
import numpy as np
from IPython.display import Markdown, display


# ============================================================
# PROJECT PATH AND MODULE RELOAD
# ============================================================

# Notebook assumed to be inside examples/
ROOT = Path.cwd().resolve().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.simulate_pursuit_evasion as pursuit

# Load the latest version of the .py file.
importlib.reload(pursuit)


# ============================================================
# METRICS
# ============================================================

METRICS = [
    "euclidean",
    "ttc",
    "dce",
]

ego_initial_state = np.array([
    0.0,  # x_E [m]
    0.0,  # y_E [m]
    0.0,  # psi_E [rad]
    0.0,  # delta_E [rad]
    4.0,  # v_E [m/s]
])

human_initial_state = np.array([
    8.0,   # x_H [m]
    -1.0,  # y_H [m]
    0.0,   # psi_H [rad]
    5.0,   # v_H [m/s]
])


# ============================================================
# SIMULATIONS AND ANIMATIONS
# ============================================================

for metric in METRICS:
    # Release memory used by the preceding metric.
    gc.collect()
    jax.clear_caches()

    display(
        Markdown(
            f"## Pursuit-evasion simulation — {metric.upper()}"
        )
    )

    video = pursuit.simulate_pursuit_evasion(
        metric=metric,
        ego_initial_state=ego_initial_state,
        human_initial_state=human_initial_state,
        recovery=True,
        T=10.0,
        dt=1 / 50,
        animation_time_scale_factor=2.0,
    )

    display(video)

    # The video has already been embedded in the notebook output.
    del video

    gc.collect()
    jax.clear_caches()

## Pursuit-evasion simulation — EUCLIDEAN

/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()
findfont: Failed to find font weight semibold, now using 700.


## Pursuit-evasion simulation — TTC


RECOVERY MODE ACTIVATED
Simulation time: 2.860000 s
Simulation step: 143
Outside dimensions:
  x_rel = 17.01201248, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 2.04999900 m/s


RECOVERY MODE ACTIVATED
Simulation time: 7.520000 s
Simulation step: 376
Outside dimensions:
  x_rel = 17.00224113, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 1.13999259 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## Pursuit-evasion simulation — DCE

/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


# BRT euclide e dce vicini allo 0, ttc positivo

In [3]:
import gc
import importlib
import sys
from pathlib import Path

import jax
import numpy as np
from IPython.display import Markdown, display


# ============================================================
# PROJECT PATH AND MODULE RELOAD
# ============================================================

# Notebook assumed to be inside examples/
ROOT = Path.cwd().resolve().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.simulate_pursuit_evasion as pursuit

# Load the latest version of the .py file.
importlib.reload(pursuit)


# ============================================================
# METRICS
# ============================================================

METRICS = [
    "euclidean",
    "ttc",
    "dce",
]

ego_initial_state = np.array([
    0.0,  # x_E [m]
    0.0,  # y_E [m]
    0.0,  # psi_E [rad]
    0.0,  # delta_E [rad]
    4.0,  # v_E [m/s]
])

human_initial_state = np.array([
    6.0,   # x_H [m]
    -1.0,  # y_H [m]
    -np.pi/6,  # psi_H [rad]
    5.0,   # v_H [m/s]
])


# ============================================================
# SIMULATIONS AND ANIMATIONS
# ============================================================

for metric in METRICS:
    # Release memory used by the preceding metric.
    gc.collect()
    jax.clear_caches()

    display(
        Markdown(
            f"## Pursuit-evasion simulation — {metric.upper()}"
        )
    )

    video = pursuit.simulate_pursuit_evasion(
        metric=metric,
        ego_initial_state=ego_initial_state,
        human_initial_state=human_initial_state,
        recovery=True,
        T=10.0,
        dt=1 / 50,
        animation_time_scale_factor=2.0,
    )

    display(video)

    # The video has already been embedded in the notebook output.
    del video

    gc.collect()
    jax.clear_caches()

## Pursuit-evasion simulation — EUCLIDEAN


RECOVERY MODE ACTIVATED
Simulation time: 4.680000 s
Simulation step: 234
Outside dimensions:
  y_rel = -8.02420616, grid = [-8.00000000, 8.00000000]
Stored ego reference speed: 2.54999852 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## Pursuit-evasion simulation — TTC


RECOVERY MODE ACTIVATED
Simulation time: 3.000000 s
Simulation step: 150
Outside dimensions:
  y_rel = -8.03943729, grid = [-8.00000000, 8.00000000]
Stored ego reference speed: 2.99999809 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## Pursuit-evasion simulation — DCE


RECOVERY MODE ACTIVATED
Simulation time: 3.780000 s
Simulation step: 189
Outside dimensions:
  y_rel = -8.01832962, grid = [-8.00000000, 8.00000000]
Stored ego reference speed: 2.54999852 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## BRT negativo

In [4]:
import gc
import importlib
import sys
from pathlib import Path

import jax
import numpy as np
from IPython.display import Markdown, display


# ============================================================
# PROJECT PATH AND MODULE RELOAD
# ============================================================

# Notebook assumed to be inside examples/
ROOT = Path.cwd().resolve().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.simulate_pursuit_evasion as pursuit

# Load the latest version of the .py file.
importlib.reload(pursuit)


# ============================================================
# METRICS
# ============================================================

METRICS = [
    "euclidean",
    "ttc",
    "dce",
]

ego_initial_state = np.array([
    0.0,  # x_E [m]
    0.0,  # y_E [m]
    0.0,  # psi_E [rad]
    0.0,  # delta_E [rad]
    8.0,  # v_E [m/s]
])

human_initial_state = np.array([
    6.0,   # x_H [m]
    0.0,  # y_H [m]
    0,  # psi_H [rad]
    5.0,   # v_H [m/s]
])


# ============================================================
# SIMULATIONS AND ANIMATIONS
# ============================================================

for metric in METRICS:
    # Release memory used by the preceding metric.
    gc.collect()
    jax.clear_caches()

    display(
        Markdown(
            f"## Pursuit-evasion simulation — {metric.upper()}"
        )
    )

    video = pursuit.simulate_pursuit_evasion(
        metric=metric,
        ego_initial_state=ego_initial_state,
        human_initial_state=human_initial_state,
        recovery=True,
        T=10.0,
        dt=1 / 50,
        animation_time_scale_factor=2.0,
    )

    display(video)

    # The video has already been embedded in the notebook output.
    del video

    gc.collect()
    jax.clear_caches()

## Pursuit-evasion simulation — EUCLIDEAN


RECOVERY MODE ACTIVATED
Simulation time: 3.040000 s
Simulation step: 152
Outside dimensions:
  x_rel = -12.01601028, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 10.99002743 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## Pursuit-evasion simulation — TTC


RECOVERY MODE ACTIVATED
Simulation time: 3.660000 s
Simulation step: 183
Outside dimensions:
  x_rel = -12.02078915, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 5.13002729 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()


## Pursuit-evasion simulation — DCE


RECOVERY MODE ACTIVATED
Simulation time: 2.980000 s
Simulation step: 149
Outside dimensions:
  x_rel = -12.16134930, grid = [-12.00000000, 17.00000000]
Stored ego reference speed: 11.00000000 m/s



/home/alessio-grisorio/TESI/hj_reachability/scripts/simulate_pursuit_evasion.py:515: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  figure.tight_layout()
